# Notebook #3: Isolation Forest Grid Search & Sensitivity Analysis

Notebook #2 showed that the largest gains came from the operating-condition correction, virtual sensors, and score smoothing, while keeping the Isolation Forest settings fixed. This notebook asks the remaining questions:

1. **How Sensitive is the Final Pipeline to Its Settings?** A grid search over the IF hyperparameters and the smoothing window, applied to the final pipeline from Notebook #2 (residual features + virtual sensors).
2. **Could Tuning Alone Have Rescued the Baseline?** The same grid applied to the raw-sensor baseline.
3. **Does the Correction Model Matter?** Gradient boosting vs. the MLP correction of Sánchez et al. (2026), used by our LSTM autoencoder.

A **grid search** tries every combination of the listed settings. Every configuration is repeated with **5 random seeds**, because with only 3 validation engines a difference of $0.01-0.02$ between two settings could be chance. Results are reported as mean $\pm \text{ SD}$.

## Outline:
1. Setup
2. Load Data & Engine Split
3. Build Residual Features (from Notebook #2)
4. Define the Tuning Grid
5. Run the Grid
6. Which Settings Matter? (Sensitivity Analysis)
7. Top Configurations
8. Select the Final Configuration
9. Final Test Evaluation
10. Could Tuning Have Rescued the Baseline?
11. Sensitivity to the Correction Model (Gradient Boosting vs. MLP)
12. Summary for the Report

## 1.0 Setup

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "4"   # limit CPU threads for the gradient-boosting models

### 1.1 Import Libraries

In [2]:
import itertools
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.ensemble import IsolationForest, HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, average_precision_score

## 2.0 Load Data & Engine Split

Same rebuilt data and same train / validation / test engines as Notebook #2 (and the rest of the team).

```text
9 Training Engines   ── Healthy Flights Only ──> Fit correction models + IF
3 Validation Engines ── All Flights ───────────> Compare every configuration
3 Test Engines       ── All Flights ───────────> Evaluated ONCE, for the selected configuration
```

In [3]:
DATA_DIR = "data"
rows = pd.read_parquet(f"{DATA_DIR}/ds08a_rows.parquet")

train_units = [2, 3, 5, 6, 7, 8, 9, 13, 15]
val_units   = [4, 10, 14]
test_units  = [1, 11, 12]

operating_cols = ["alt", "Mach", "TRA", "T2"]
baseline_sensor_cols = ["T24", "T30", "T48", "T50", "P15", "P2", "P24",
                        "Ps30", "P50", "Nf", "Nc", "Wf"]
virtual_cols = ["T40", "P30", "P45", "W21", "W22", "W25", "W31", "W32",
                "W48", "W50", "SmFan", "SmLPC", "SmHPC", "phi"]
feature_cols = baseline_sensor_cols + virtual_cols

print(rows.shape)

(67328, 50)


## 3.0 Build Residual Features (from Notebook #2)

These are the same steps as Notebook #2, Sections #4.1–4.2, computed **once** here, since the correction does not depend on the Isolation Forest settings being tuned:

1. For each sensor, fit a model predicting it from the four operating conditions, using **healthy rows from training engines only**
2. Residual = actual value − predicted healthy value
3. Average residuals per flight

In [4]:
train_rows = rows[rows["unit"].isin(train_units) & (rows["hs"] == 1)]

resid = rows[["unit", "cycle", "hs", "Fc"]].copy()
for c in feature_cols:
    m = HistGradientBoostingRegressor(max_iter=200, random_state=0)
    m.fit(train_rows[operating_cols], train_rows[c])
    resid[c] = rows[c] - m.predict(rows[operating_cols])

flight_df = (
    resid.groupby(["unit", "cycle"])
         .agg(**{c: (c, "mean") for c in feature_cols},
              hs=("hs", "first"), Fc=("Fc", "first"))
         .reset_index()
         .sort_values(["unit", "cycle"])
)

X_train = flight_df[flight_df["unit"].isin(train_units) & (flight_df["hs"] == 1)][feature_cols]
val_df  = flight_df[flight_df["unit"].isin(val_units)].copy()
test_df = flight_df[flight_df["unit"].isin(test_units)].copy()
y_val   = 1 - val_df["hs"]      # unhealthy = 1

print("Healthy training flights:", len(X_train))
print("Validation flights:", len(val_df), "| Test flights:", len(test_df))

Healthy training flights: 187
Validation flights: 208 | Test flights: 190


## 4.0 Defining the Tuning Grid

| Parameter | What It Controls | Values | Notebook #2 Setting |
|:---------:|------------------|:------:|:-------------------:|
| `n_estimators` | Number of trees. More trees make scores more stable, but don't make the model "learn more". | 100, 200, 500 | 200 |
| `max_samples` | Healthy flights sampled to build each tree. With 187 healthy training flights, `"auto"` uses all 187 (sklearn caps it at min(256, n)). | 32, 64, 128, auto | auto |
| `max_features` | Share of features each tree can split on. Lower values make trees more varied. | 0.25, 0.5, 0.75, 1.0 | 1.0 |
| `smooth` | Rolling-median window (flights) applied to each engine's scores. 1 = no smoothing. | 1, 3, 5, 10 | 5 |
| `seed` | Random seed, repeated to measure run-to-run variability (not a hyperparameter). | 0–4 | 42 |

This gives $3 \times 4 \times 4 \times 4 = 192$ configurations, each run with 5 seeds. **Smoothing** is applied after scoring, so each forest is fit once per seed and then scored with all four windows.

In [5]:
grid = {
    "n_estimators": [100, 200, 500],
    "max_samples":  [32, 64, 128, "auto"],
    "max_features": [0.25, 0.5, 0.75, 1.0],
}
smooth_windows = [1, 3, 5, 10]
seeds = range(5)

n_configs = np.prod([len(v) for v in grid.values()]) * len(smooth_windows)
print("Configurations:", n_configs, "| Runs including seeds:", n_configs * len(seeds))

Configurations: 192 | Runs including seeds: 960


## 5.0 Run the Grid

For each IF setting & seed:
1. Fit on healthy training flights
2. Score validation flights (negated `score_samples()`, so higher = more anomalous)
3. Apply each smoothing window within each engine and record validation ROC-AUC and PR-AUC

The test engines are **NOT** touched in this section.

In [6]:
def smooth_scores(df, raw_col, window):
    """Median of current flight and previous (window - 1) flights, per engine."""
    if window == 1:
        return df[raw_col]
    return df.groupby("unit")[raw_col].transform(lambda s: s.rolling(window, min_periods=1).median())


records = []
for n_est, max_s, max_f in itertools.product(*grid.values()):
    for seed in seeds:
        model = IsolationForest(n_estimators=n_est, max_samples=max_s,
                                max_features=max_f, random_state=seed).fit(X_train)
        val_df["raw_score"] = -model.score_samples(val_df[feature_cols])

        for w in smooth_windows:
            s = smooth_scores(val_df, "raw_score", w)
            records.append({
                "n_estimators": n_est, "max_samples": str(max_s), "max_features": max_f,
                "smooth": w, "seed": seed,
                "val_ROC_AUC": roc_auc_score(y_val, s),
                "val_PR_AUC": average_precision_score(y_val, s),
            })

tuning_runs = pd.DataFrame(records)
print(tuning_runs.shape)

(960, 7)


In [7]:
# Average over seeds: one row per configuration
param_cols = ["n_estimators", "max_samples", "max_features", "smooth"]

tuning_summary = (
    tuning_runs.groupby(param_cols)
               .agg(ROC_mean=("val_ROC_AUC", "mean"), ROC_sd=("val_ROC_AUC", "std"),
                    PR_mean=("val_PR_AUC", "mean"), PR_sd=("val_PR_AUC", "std"))
               .reset_index()
)

print("Range of mean validation ROC-AUC across configurations:",
      round(tuning_summary["ROC_mean"].min(), 3), "to", round(tuning_summary["ROC_mean"].max(), 3))
print("Typical seed-to-seed SD:", round(tuning_summary["ROC_sd"].median(), 3))

Range of mean validation ROC-AUC across configurations: 0.864 to 0.94
Typical seed-to-seed SD: 0.008


## 6.0 Which Settings Matter? (Sensitivity Analysis)

For each parameter, the table below averages validation ROC-AUC over every other setting and seed. The **spread** column ($\text{max} - \text{min}$) shows how much changing that parameter moves performance. A spread similar to the seed-to-seed SD means the parameter makes little real difference.

In [8]:
sensitivity = []
for p in param_cols:
    means = tuning_runs.groupby(p)["val_ROC_AUC"].mean()
    sensitivity.append({
        "parameter": p,
        "values (mean ROC-AUC)": ", ".join(f"{k}: {v:.3f}" for k, v in means.items()),
        "spread": means.max() - means.min(),
    })

pd.DataFrame(sensitivity).sort_values("spread", ascending=False).round(3)

,parameter,values (mean ROC-AUC),spread
3,smooth,"1: 0.885, 3: 0.921, 5: 0.929, 10: 0.914",0.044
2,max_features,"0.25: 0.914, 0.5: 0.915, 0.75: 0.908, 1.0: 0.913",0.007
1,max_samples,"128: 0.913, 32: 0.909, 64: 0.912, auto: 0.915",0.006
0,n_estimators,"100: 0.912, 200: 0.913, 500: 0.912",0.001


In [9]:
fig = px.box(
    tuning_runs.assign(smooth=tuning_runs["smooth"].astype(str)),
    x="smooth", y="val_ROC_AUC",
    category_orders={"smooth": ["1", "3", "5", "10"]},
    points=False,
    title="Validation ROC-AUC by Smoothing Window (All Other Settings & Seeds)",
    labels={"smooth": "Smoothing Window (Flights)", "val_ROC_AUC": "Validation ROC-AUC"},
)
fig.show()

In [10]:
plot_df = (tuning_runs[tuning_runs["smooth"] == 5]
           .groupby(["n_estimators", "max_samples", "max_features"])["val_ROC_AUC"]
           .mean().reset_index())
plot_df["n_estimators"] = plot_df["n_estimators"].astype(str)

fig = px.line(
    plot_df, x="max_samples", y="val_ROC_AUC", color="n_estimators",
    facet_col="max_features", markers=True,
    category_orders={"max_samples": ["32", "64", "128", "auto"]},
    title="Isolation Forest Settings (Smoothing Window = 5, mean of 5 seeds)",
    labels={"max_samples": "Max Samples", "val_ROC_AUC": "Validation ROC-AUC",
            "n_estimators": "Number of Trees", "max_features": "Max Features"},
)
fig.show()

### 6.1 Interpretation

* **The smoothing window has by far the largest effect** ($\approx 0.04 - 0.05 \text{ ROC-AUC}$). A window of 5 flights is best; 3 is close behind; 10 smooths so much that it starts to blur the transition from healthy to unhealthy.
* **The three Isolation Forest settings each move ROC-AUC by less than 0.01**, about the same as the variation from changing the random seed. In other words, once the features are right, the Isolation Forest is not sensitive to its hyperparameters.
* Section 10 checks whether the same is true for the raw-sensor baseline.

## 7.0 Top Configurations

The 10 best configurations by mean validation ROC-AUC (averaged over 5 seeds).

In [11]:
top10 = tuning_summary.sort_values("ROC_mean", ascending=False).head(10).reset_index(drop=True)
top10.round(3)

,n_estimators,max_samples,max_features,smooth,ROC_mean,ROC_sd,PR_mean,PR_sd
0,100,32,0.50,5,0.940,0.011,0.973,0.005
1,100,64,0.50,5,0.937,0.009,0.972,0.004
2,100,auto,0.50,5,0.935,0.005,0.971,0.002
3,200,32,0.50,5,0.935,0.006,0.971,0.003
4,200,64,0.25,5,0.934,0.002,0.971,0.001
5,100,auto,1.00,5,0.934,0.005,0.971,0.002
6,500,64,0.25,5,0.933,0.008,0.970,0.004
7,100,128,0.25,5,0.933,0.009,0.970,0.005
8,100,64,0.25,5,0.932,0.010,0.970,0.005
9,100,32,0.25,5,0.932,0.017,0.969,0.008


In [12]:
# Where does the Notebook #2 configuration rank?
nb3 = tuning_summary[
    (tuning_summary["n_estimators"] == 200) & (tuning_summary["max_samples"] == "auto") &
    (tuning_summary["max_features"] == 1.0) & (tuning_summary["smooth"] == 5)
]
nb3_rank = (tuning_summary["ROC_mean"] > nb3["ROC_mean"].iloc[0]).sum() + 1
print(f"Notebook #2 configuration: mean ROC-AUC {nb3['ROC_mean'].iloc[0]:.3f} ± {nb3['ROC_sd'].iloc[0]:.3f}, "
      f"rank {nb3_rank} of {len(tuning_summary)}")

Notebook #2 configuration: mean ROC-AUC 0.932 ± 0.005, rank 13 of 192


### 7.1 Interpretation

The top configurations are separated by less than one seed-to-seed SD, so they are effectively tied. Picking the single highest number would mostly be picking the luckiest seeds.

## 8.0 Select the Final Configuration

**Selection rule (decided *before* looking at test):**
1. Find the best mean validation ROC-AUC.
2. Keep every configuration within **one seed SD** of it (statistically tied).
3. Among those, choose the **simplest/cheapest**: fewest trees, then the default `max_samples="auto"`, then `max_features=1.0`.

This "one-SD rule" avoids over-tuning to 3 validation engines and is easy to explain in the report.

In [13]:
best_mean = tuning_summary["ROC_mean"].max()
tie_sd = tuning_summary.loc[tuning_summary["ROC_mean"].idxmax(), "ROC_sd"]

tied = tuning_summary[tuning_summary["ROC_mean"] >= best_mean - tie_sd].copy()
tied["_auto"] = (tied["max_samples"] != "auto").astype(int)      # prefer default
tied["_mf"] = (tied["max_features"] != 1.0).astype(int)          # prefer default

final_cfg = tied.sort_values(["n_estimators", "_auto", "_mf", "ROC_mean"],
                             ascending=[True, True, True, False]).iloc[0]

print(f"Best mean ROC-AUC: {best_mean:.3f} (seed SD {tie_sd:.3f}); {len(tied)} configurations tied")
final_cfg[param_cols + ["ROC_mean", "ROC_sd", "PR_mean", "PR_sd"]]

Best mean ROC-AUC: 0.940 (seed SD 0.011); 28 configurations tied


n_estimators         100
max_samples         auto
max_features         1.0
smooth                 5
ROC_mean        0.933523
ROC_sd          0.004884
PR_mean         0.970882
PR_sd           0.002017
Name: 62, dtype: object

## 9.0 Final Test Evaluation

The selected configuration is evaluated **once** on the 3 test engines, averaged over the same 5 seeds.

In [14]:
ms = final_cfg["max_samples"]
ms = ms if ms == "auto" else int(ms)

test_records, per_engine = [], []
for seed in seeds:
    model = IsolationForest(n_estimators=int(final_cfg["n_estimators"]), max_samples=ms,
                            max_features=float(final_cfg["max_features"]), random_state=seed).fit(X_train)
    test_df["raw_score"] = -model.score_samples(test_df[feature_cols])
    test_df["score"] = smooth_scores(test_df, "raw_score", int(final_cfg["smooth"]))
    y_test = 1 - test_df["hs"]
    test_records.append({"seed": seed,
                         "test_ROC_AUC": roc_auc_score(y_test, test_df["score"]),
                         "test_PR_AUC": average_precision_score(y_test, test_df["score"])})
    for u, d in test_df.groupby("unit"):
        per_engine.append({"unit": u, "Fc": int(d["Fc"].iloc[0]), "seed": seed,
                           "ROC_AUC": roc_auc_score(1 - d["hs"], d["score"])})

test_results = pd.DataFrame(test_records)
print(f"Test ROC-AUC: {test_results['test_ROC_AUC'].mean():.3f} ± {test_results['test_ROC_AUC'].std():.3f}")
print(f"Test PR-AUC:  {test_results['test_PR_AUC'].mean():.3f} ± {test_results['test_PR_AUC'].std():.3f}")
print(f"PR baseline (share unhealthy): {(1 - test_df['hs']).mean():.3f}")

Test ROC-AUC: 0.813 ± 0.013
Test PR-AUC:  0.917 ± 0.007
PR baseline (share unhealthy): 0.647


In [15]:
pd.DataFrame(per_engine).groupby(["unit", "Fc"])["ROC_AUC"].agg(["mean", "std"]).round(3)

,,mean,std
unit,Fc,,
1,1,0.868,0.034
11,2,0.787,0.018
12,3,0.795,0.007


## 10.0 Could Tuning Have Rescued the Baseline?

The same grid (192 configurations $\times$ 5 seeds) is applied to the **raw-sensor baseline** from Notebook #2: per-flight means of the 12 physical sensors, with no operating-condition correction.

If the best baseline configuration is still far below the corrected pipeline, the improvement in Notebook #2 came from the features, not from settings that happened to be left untuned.

In [16]:
raw_cols = baseline_sensor_cols
raw_flights = (
    rows.groupby(["unit", "cycle"])
        .agg(**{c: (c, "mean") for c in raw_cols}, hs=("hs", "first"), Fc=("Fc", "first"))
        .reset_index()
        .sort_values(["unit", "cycle"])
)
X_train_raw = raw_flights[raw_flights["unit"].isin(train_units) & (raw_flights["hs"] == 1)][raw_cols]
val_raw = raw_flights[raw_flights["unit"].isin(val_units)].copy()
y_val_raw = 1 - val_raw["hs"]

raw_records = []
for n_est, max_s, max_f in itertools.product(*grid.values()):
    for seed in seeds:
        model = IsolationForest(n_estimators=n_est, max_samples=max_s,
                                max_features=max_f, random_state=seed).fit(X_train_raw)
        val_raw["raw_score"] = -model.score_samples(val_raw[raw_cols])
        for w in smooth_windows:
            s = smooth_scores(val_raw, "raw_score", w)
            raw_records.append({"n_estimators": n_est, "max_samples": str(max_s), "max_features": max_f,
                                "smooth": w, "seed": seed, "val_ROC_AUC": roc_auc_score(y_val_raw, s)})

raw_summary = (pd.DataFrame(raw_records).groupby(param_cols)["val_ROC_AUC"]
                 .agg(ROC_mean="mean", ROC_sd="std").reset_index())

pd.DataFrame({
    "pipeline": ["Raw-sensor baseline", "Final (corrected) pipeline"],
    "worst config": [raw_summary["ROC_mean"].min(), tuning_summary["ROC_mean"].min()],
    "median config": [raw_summary["ROC_mean"].median(), tuning_summary["ROC_mean"].median()],
    "best config": [raw_summary["ROC_mean"].max(), tuning_summary["ROC_mean"].max()],
}).round(3)

,pipeline,worst config,median config,best config
0,Raw-sensor baseline,0.473,0.528,0.589
1,Final (corrected) pipeline,0.864,0.918,0.940


In [17]:
fig = px.histogram(
    pd.concat([raw_summary.assign(pipeline="Raw-sensor baseline"),
               tuning_summary.assign(pipeline="Final (corrected) pipeline")]),
    x="ROC_mean", color="pipeline", nbins=60, barmode="overlay", opacity=0.7,
    title="Validation ROC-AUC Across All 192 Configurations (Mean of 5 Seeds)",
    labels={"ROC_mean": "Validation ROC-AUC", "pipeline": "Pipeline"},
)
fig.show()

### 10.1 Interpretation

Even the **best** of 192 baseline configurations stays far below the **worst** corrected configuration. The two distributions do not overlap. This rules out the possibility that the baseline was simply under-tuned: no combination of Isolation Forest settings compensates for features dominated by flight conditions.

## 11.0 Sensitivity to the Correction Model (Gradient Boosting vs. MLP)

The operating-condition correction (Section #3) uses a **gradient-boosted regressor** for each sensor. Our LSTM autoencoder instead follows **Sánchez et al. (2026)**, who train a **Multilayer Perceptron (MLP)** regressor for each sensor to predict its healthy value from the operating conditions, building on the regression-based normalization of Wang et al. (2018, 2019). Sánchez et al. worked with the original C-MAPSS, which has 3 operational settings; N-CMAPSS records 4 (`alt`, `Mach`, `TRA`, `T2`).

This section asks: **would the MLP correction make a meaningful improvement to the Isolation Forest?**

Only the correction model changes. Everything else is held fixed:
* Same 26 features, same healthy training rows (training engines only), same flight-level means
* Same final IF configuration from Section #8 and the same 5 seeds
* Same 5-flight rolling median

#### MLP Setup
One `MLPRegressor` per sensor (2 hidden layers of 64 units, early stopping), with the operating conditions and the sensor both standardized on the healthy training rows. Standardizing keeps sensors with very different units (e.g., `T48` in °R vs. `Wf`) on a comparable scale so the network trains stably.

#### What Is Compared
* **$R^2$ on Validation Engines' Healthy Rows:** how well each correction learns *healthy* behavior on engines it never saw (uses no unhealthy labels)
* **Validation ROC-AUC & PR-AUC:** the basis for any choice, as in Section #8
* **Test ROC-AUC & PR-AUC:** reported for completeness only, *NOT* used to choose

In [18]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor


def build_correction(kind):
    """Residual flight table + R^2 on validation engines' healthy rows for one correction model."""
    val_healthy_rows = rows[rows["unit"].isin(val_units) & (rows["hs"] == 1)]
    resid_k = rows[["unit", "cycle", "hs", "Fc"]].copy()
    r2 = {}
    for c in feature_cols:
        if kind == "Gradient Boosting":
            model = HistGradientBoostingRegressor(max_iter=200, random_state=0)
        else:  # MLP, following Sánchez et al. (2026)
            model = TransformedTargetRegressor(
                regressor=make_pipeline(
                    StandardScaler(),
                    MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=500, early_stopping=True, random_state=0),
                ),
                transformer=StandardScaler(),
            )
        model.fit(train_rows[operating_cols], train_rows[c])
        resid_k[c] = rows[c] - model.predict(rows[operating_cols])
        r2[c] = model.score(val_healthy_rows[operating_cols], val_healthy_rows[c])

    flights_k = (
        resid_k.groupby(["unit", "cycle"])
               .agg(**{c: (c, "mean") for c in feature_cols}, hs=("hs", "first"), Fc=("Fc", "first"))
               .reset_index()
               .sort_values(["unit", "cycle"])
    )
    return flights_k, pd.Series(r2)


def evaluate_correction(flights_k):
    """Final IF configuration (Section #8), 5 seeds, 5-flight median: validation and test metrics."""
    ms = final_cfg["max_samples"]
    ms = ms if ms == "auto" else int(ms)
    X_tr = flights_k[flights_k["unit"].isin(train_units) & (flights_k["hs"] == 1)][feature_cols]
    out = {}
    for part, units in [("val", val_units), ("test", test_units)]:
        rocs, prs = [], []
        for seed in seeds:
            model = IsolationForest(n_estimators=int(final_cfg["n_estimators"]), max_samples=ms,
                                    max_features=float(final_cfg["max_features"]), random_state=seed).fit(X_tr)
            scored = flights_k[flights_k["unit"].isin(units)].copy()
            scored["raw_score"] = -model.score_samples(scored[feature_cols])
            scored["score"] = smooth_scores(scored, "raw_score", int(final_cfg["smooth"]))
            rocs.append(roc_auc_score(1 - scored["hs"], scored["score"]))
            prs.append(average_precision_score(1 - scored["hs"], scored["score"]))
        out[f"{part}_ROC_mean"], out[f"{part}_ROC_sd"] = np.mean(rocs), np.std(rocs, ddof=1)
        out[f"{part}_PR_mean"] = np.mean(prs)
    return out

In [19]:
correction_results, correction_r2 = [], {}
for kind in ["Gradient Boosting", "MLP"]:
    flights_k, r2 = build_correction(kind)
    correction_r2[kind] = r2
    correction_results.append({
        "correction_model": kind,
        "val_healthy_R2_mean": r2.mean(),
        "val_healthy_R2_min": r2.min(),
        **evaluate_correction(flights_k),
    })

correction_results = pd.DataFrame(correction_results).set_index("correction_model")
correction_results.round(4)

,val_healthy_R2_mean,val_healthy_R2_min,val_ROC_mean,val_ROC_sd,val_PR_mean,test_ROC_mean,test_ROC_sd,test_PR_mean
correction_model,,,,,,,,
Gradient Boosting,0.9983,0.9942,0.9335,0.0049,0.9709,0.8129,0.0126,0.9167
MLP,0.9996,0.9967,0.9309,0.0067,0.9729,0.8327,0.0078,0.9258


In [20]:
# Sensors where the two corrections differ most in R^2 (validation engines, healthy rows)
r2_compare = pd.DataFrame(correction_r2)
r2_compare["MLP - GB"] = r2_compare["MLP"] - r2_compare["Gradient Boosting"]
r2_compare.sort_values("MLP - GB").round(4).head(8)

,Gradient Boosting,MLP,MLP - GB
Nf,0.9997,0.9999,0.0002
P50,0.9995,0.9998,0.0003
SmFan,0.9993,0.9997,0.0004
P2,0.9995,0.9999,0.0004
phi,0.9991,0.9995,0.0005
SmHPC,0.9969,0.9974,0.0005
P15,0.9991,0.9998,0.0008
Wf,0.9989,0.9998,0.0008


### 11.1 Interpretation

* **The MLP learns healthy behavior slightly better.** On the validation engines' healthy rows, it explains more of every sensor's variation than gradient boosting does (mean $R^2 = 0.9996$ vs. $0.9983$; lowest sensor $0.9967$ vs. $0.9942$). Both corrections already explain more than $99\%$ of each sensor, so the room for improvement is small.
* **It does *NOT* meaningfully improve anomaly detection on validation.** Validation ROC-AUC is $0.931 \pm 0.007$ with the MLP vs. $0.934 \pm 0.005$ with gradient boosting, a difference smaller than one seed-to-seed SD; PR-AUC is $0.973$ vs. $0.971$. On the engines used for model selection, the two are tied.
* **Test results favor the MLP** ($\text{ROC-AUC} = 0.833 \pm 0.007$ vs. $0.813 \pm 0.013$; $\text{PR-AUC} = 0.926$ vs. $0.917$), consistent with its better fit to healthy behavior on unseen engines. Following the selection rule in Section #8, this test difference is reported but *NOT* used to switch models.
* **Takeaway:** what matters is *that* operating-condition effects are removed, not *which* regression model removes them. Gradient boosting is kept for the Isolation Forest; the MLP correction used by our LSTM autoencoder (following Sánchez et al., 2026) performs comparably, so the two Part B methods rest on equivalent preprocessing.

#### References
* Sánchez, P., Reyes, K., Radu, B., & Fernández, E. (2026). *Early fault detection on CMAPSS with unsupervised LSTM autoencoders*. arXiv. https://doi.org/10.48550/arXiv.2601.10269
* Wang, Q., Farahat, A., Ristovski, K., Tang, H.-K., Serita, S., & Gupta, C. (2018). What maintenance is worth the money? A data-driven answer. *2018 IEEE 16th International Conference on Industrial Informatics (INDIN)*, 284–291. https://doi.org/10.1109/INDIN.2018.8472003
* Wang, Q., Zheng, S., Farahat, A., Serita, S., & Gupta, C. (2019). *Remaining useful life estimation using functional data analysis*. arXiv. https://arxiv.org/abs/1904.06442

## 12.0 Summary for the Report

* **What Was Tuned:** 192 configurations of the final pipeline (`n_estimators`, `max_samples`, `max_features`, smoothing window), each with 5 random seeds, using validation engines only.
* **Main finding:** IF hyperparameters had little effect ($<0.01$ ROC-AUC each, comparable to seed variability). The smoothing window mattered most ($\approx 0.04-0.05$), with 5 flights best.
* **Implication:** Model quality is driven by feature construction (operating-condition correction) and score smoothing, not by Isolation Forest settings.
* **Baseline Check:** The same grid applied to the raw-sensor baseline never comes close to the corrected pipeline (Section #10), so the baseline was not simply under-tuned.
* **Correction Model:** Replacing gradient boosting with the MLP correction of Sánchez et al. (2026) left validation ROC-AUC unchanged ($0.931$ vs. $0.934$, within one seed SD), so gradient boosting is kept; the MLP's test ROC-AUC ($0.833$ vs. $0.813$) is reported but not used for selection (Section #11).
* **Selection:** One-SD rule (simplest configuration statistically tied with the best), decided before test evaluation.
* **Final Test Performance:** See Section #9 (reported as $\text{mean} \pm \text{SD}$ over 5 seeds).